In [1]:
import os 
from langchain.chat_models import init_chat_model
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY") 
from dotenv import load_dotenv
load_dotenv()

llm = ChatGroq(model = "llama-3.3-70b-versatile")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000025D672A8830>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025D672A9550>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from pydantic import BaseModel, Field
from groq import RateLimitError

class Movie(BaseModel):
    title:str = Field(description= "The title of the movie")
    year:int = Field(description= "This year the movie was released")
    director: str = Field(description= "The director of the movie")
    rating: float = Field(description= "The movies rating out of 10")

In [3]:
llm_with_structured_output = llm.with_structured_output(Movie)
llm_with_structured_output

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000025D672A8830>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025D672A9550>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}}, 'required': ['title',

In [4]:
llm.invoke("Provide details about the movie Inception")

AIMessage(content='**Inception (2010) - A Mind-Bending Sci-Fi Action Film**\n\nDirected by Christopher Nolan, Inception is a thought-provoking and visually stunning movie that delves into the concept of shared dreaming. The film features an ensemble cast, including Leonardo DiCaprio, Joseph Gordon-Levitt, Ellen Page, Tom Hardy, and Marion Cotillard.\n\n**Plot**\n\nThe movie follows Cobb (Leonardo DiCaprio), a skilled thief who specializes in entering people\'s dreams and stealing their secrets. Cobb is hired by a wealthy businessman named Saito (Ken Watanabe) to perform a task known as "inception" - planting an idea in someone\'s mind instead of stealing one.\n\nSaito wants Cobb to convince Robert Fischer (Cillian Murphy), the son of a dying business magnate, to dissolve his father\'s company. In return, Saito promises to clear Cobb\'s name, which is wanted by the authorities, and allow him to return to the United States to see his children.\n\nCobb assembles a team of experts, includi

In [5]:
llm_with_structured_output.invoke("Provide details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.5)

## Nested Structure

In [11]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name : str
    role : str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description= "Budget in millions USD")

llm_with_structure = llm.with_structured_output(MovieDetails)

response = llm_with_structure.invoke("Provide details about the movie The Crow")
response

MovieDetails(title='The Crow', year=1994, cast=[Actor(name='Brandon Lee', role='Eric Draven'), Actor(name='Rochelle Davis', role='Sarah')], genres=['Action', 'Fantasy', 'Thriller'], budget=23.0)

Temel olarak Nested Structure, bir veri modelinin içinde başka bir veri modelini barındırmasıdır. 
MovieDetails sınıfının içindeki cast listesi, aslında Actor tipindeki objelerden oluşuyor.

1. Nested Structure Nedir?
Yazılımda ve veri biliminde "nested" (iç içe), bir veri yapısının başka bir veri yapısını içermesi demektir. Pydantic özelinde konuşursak; bir BaseModel sınıfının, başka bir BaseModel sınıfını bir tip (type) olarak kullanmasıdır.

2. Neden Kullanılır?
Veriyi gerçek dünyadaki karmaşıklığına uygun şekilde organize etmek için kullanılır.

* Hiyerarşi Kurmak: Bir film sadece metinlerden oluşmaz; içinde oyuncular, teknik ekip veya ödüller gibi alt gruplar barındırır.

* Kodun Tekrar Kullanılabilirliği (Reusability): Actor sınıfını bir kez tanımlarsın; sonra bunu Movie, TheaterPlay veya Commercial sınıflarının içinde tekrar tekrar kullanabilirsin.

* Okunabilirlik: Tüm veriyi tek bir devasa sınıfa yığmak yerine, parçalara bölerek yönetmeyi sağlar.

## TypeDict

TypedDict, çalışma zamanı doğrulamasına (runtime validation) ihtiyaç duymadığınız durumlar için ideal, Python'un yerleşik yazım (typing) sistemini kullanan daha basit bir alternatif sunar.

TypedDict Nedir?
Pydantic'teki BaseModel yapısına çok benzer görünse de, TypedDict aslında sadece bir ipucu (hint) mekanizmasıdır. Python'daki standart sözlüklerin (dictionary) hangi anahtarlara (key) ve ne tür değerlere (value) sahip olması gerektiğini belirtmek için kullanılır.


Ne Zaman Kullanılır?
* Hafif Bir Yapı Lazımsa: Ekstra kütüphane yükü istemiyor, sadece "bu sözlükte name ve age anahtarları olsun" demek istiyorsan.

* Validasyon Gerekmiyorsa: Verinin zaten doğru formatta geleceğine güveniyorsan (veya performans çok kritikse).

* Mevcut Sözlüklerle Çalışırken: Elinde zaten standart sözlüklerden oluşan bir yapı varsa ve bunu bozmadan tip güvenliği eklemek istiyorsan.



In [15]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ...,  "This year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movies rating out of 10"]

llm_withtypedict = llm.with_structured_output(MovieDict)
response = llm_withtypedict.invoke("Please provide the details of the movie Requiem For a Dream")
response

{'director': 'Darren Aronofsky',
 'rating': 8.4,
 'title': 'Requiem For a Dream',
 'year': 2000}

In [16]:

class Actor(TypedDict):
    name : str
    role : str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description= "Budget in millions USD")

llm_with_structure = llm.with_structured_output(MovieDetails)

response = llm_with_structure.invoke("Provide details about the movie The Crow")
response

{'budget': 23,
 'cast': [{'name': 'Brandon Lee', 'role': 'Eric Draven'},
  {'name': 'Rochelle Davis', 'role': 'Sarah'},
  {'name': 'Ernie Hudson', 'role': 'Sergeant Albrecht'}],
 'genres': ['Action', 'Fantasy', 'Thriller'],
 'title': 'The Crow',
 'year': 1994}

## Data Class

Python'da normal bir sınıf (class) oluşturduğunuzda; objeyi yazdırmak, karşılaştırmak veya başlatmak için __init__, __repr__ ve __eq__ gibi "boilerplate" dediğimiz (sürekli tekrar eden sıkıcı kodlar) metodları tek tek yazmanız gerekir.

Data Class, bu sıkıcı işleri sizin yerinize otomatik yapan bir yapıdır. Amacı sadece veri taşımaktır.

Neden Kullanılır?
* Kod Kalabalığını Önler: Onlarca satır sürecek __init__ fonksiyonunu yazmanıza gerek kalmaz.

* Okunabilirlik: Sınıfın hangi verilere sahip olduğu ilk bakışta anlaşılır.

* Otomatik Özellikler: Nesneyi print() ile yazdırdığınızda düzgün bir formatta görünür, iki nesneyi == ile karşılaştırdığınızda içindeki verilere bakar.


* Pydantic (BaseModel): Eğer dış dünyadan (API, Kullanıcı, LLM) gelen veriyi doğrulamak (validate) ve yanlışsa hata almak istiyorsan bunu kullan. En güvenlisidir.

* TypedDict: Eğer sadece bir sözlük (dict) kullanmak istiyorsan ama kod yazarken hangi anahtarların olduğunu unutmamak için bir ipucu istiyorsan bunu kullan. Hafiftir.

* Data Class: Eğer uygulamanın içinde kendi oluşturduğun verileri saklamak için gerçek bir Python objesi istiyorsan ama __init__ yazmakla uğraşmak istemiyorsan bunu kullan.

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person.""",
    name: str : "The name of the person",
    email: str :  "The email address of the person",
    phone: str : "The phone number of the person",


agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]